In [ ]:
#TODO: описать что происходит


# Устанавливаем зависимости

In [1]:
%pip install torch transformers scikit-learn sentence-transformers datasets


# 1 Грузим датасет с дефинициями



In [2]:
from datasets import load_dataset

ds = load_dataset("snagbreac/russian-reverse-dictionary-train-data") # датасет со словарными определениями


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [3]:
print(len(ds['train']))


294007


In [4]:
print(ds['train'][1000])


{'word': 'адъютантский', 'definition': 'принадлежащий адъютанту', 'df': 'efremova'}


In [5]:
from collections import defaultdict

# индексируем определения чтобы удобнее обращаться
#TODO выгрузить чтобы не надо было каждый раз индексировать
word2defs = defaultdict(list)

for entry in ds['train']:
    word2defs[entry['word']].append(entry['definition'])


In [6]:
automat_definitions = word2defs.get('автомат')
print(automat_definitions)


['устройство, самостоятельно, без непосредственного участия человека выполняющее какие-л. действия или операции в соответствии с заранее заданной программой', 'перен. тот, кто поступает не думая, делает что-л. механически, машинально', 'перен. разг. оценка, полученная учащимся, студентом без дополнительной проверки приобретенных ими знаний и основанная на личной уверенности преподавателя в достаточности этих знаний', 'ручное автоматическое скорострельное оружие', 'разг. телефон-автомат', 'разновидность армейского оружия', 'индивидуальное огнестрельное оружие', 'устройство, производящее работу по заданной программе без непосредственного участия человека', 'аппарат, производящий некоторые действия без непосредственного участия человека', 'ручное скорострельное оружие', 'машина, после включения самостоятельно выполняющая ряд заданных операций']


# Грузим BERT

context → BERT → embedding C

gloss   → BERT → embedding G

score = cosine(C, G)

In [ ]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")
model = AutoModel.from_pretrained("DeepPavlov/rubert-base-cased", device_map="auto")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# 2 Обрабатываем конлю

## 2.1 Функции для парсинга

In [7]:
%pip install conllu


### 2.1.1 Вытаскиваем семантические теги

In [8]:
import re
SEM_PATTERN = re.compile(r"sem=\[([^\]]+)\]")

def extract_sem_tags(misc): # вытаскиваем семантические теги
    if misc is None:
        return []

    # случай: conllu вернул dict
    if isinstance(misc, dict):
        sem_val = misc.get("sem")
        if not sem_val:
            return []
        sem_val = sem_val.strip("[]")
        return [t.strip() for t in sem_val.split(",")]

    # случай: строка
    match = SEM_PATTERN.search(str(misc))
    if not match:
        return []

    return [t.strip() for t in match.group(1).split(",")]


### 2.1.2 Приводим токены CoNLL-U к формату Word2Vec НКРЯ (чтобы потом считать векторы слов и предложений)

In [9]:
pos_map = {
    "NOUN":  "S",       # существительное
    "ADJ":   "A",       # прилагательное
    "VERB":  "V",       # глагол
    "ADV":   "ADV",     # наречие
    "PRON":  "SPRO",    # местоимение
    "ADVPRO": "ADV",    # местоименное наречие
    "NUM":   "NUM",     # числительное
    "PROPN": "S",       # имя собственное → существительное

    "ADP":   "PR",      # предлог
    "CCONJ": "CONJ",    # сочинительный союз
    "SCONJ": "CONJ",    # подчинительный союз
    "PART":  "PART",    # частица
    "INTJ":  "INTJ",    # междометие

    "DET":   "APRO",    # определительное/местоименное прилагательное
    "AUX":   "V",       # вспомогательный глагол
    "SYM":   "SYM",     # символ
    "X":     "X",       # другое / неизвестное
    }

def conllu_to_w2v_format(lemma, upos): # POS MyStem → POS UD
    pos = pos_map.get(upos)

    if pos is None:
        return lemma

    return f"{lemma}_{pos}"

# TODO: посмотреть отдельные слова, например, здесь ADVPRO не переводится в ADV
# [w for w in wv.key_to_index if w.startswith("лингвистика_")][:20]


### 2.1.3 Парсим конлю и приводим в нужный формат

In [10]:
from conllu import parse

def parse_conllu(text): # берем предложение/текст и делаем словарь из словарей с признаками слова
    sentences = parse(text)
    parsed = []

    for sent in sentences:
        tokens = []
        for token in sent:
            sem_tags = extract_sem_tags(token.get("misc"))
            UD_lemma = conllu_to_w2v_format(token["lemma"], token["upos"])

            tokens.append({
                "form": token["form"],
                "lemma": token["lemma"],
                "upos": token["upos"],
                "UD_lemma": UD_lemma,
                "UD_pos": pos_map.get(token["upos"]),
                "feats": token["feats"],
                "head": token["head"],
                "deprel": token["deprel"],
                "sem": sem_tags
            })

        parsed.append(tokens)

    return parsed

# выдает набор токенов в таком формате:
# {'form': 'С', 'lemma': 'с', 'upos': 'ADP', 'UD_lemma': 'с_PR', 'UD_pos': 'PR', 'feats': None, 'head': 3, 'deprel': 'case', 'sem': []}


## 2.2 Загружаем конлю данные

In [11]:
conllu_file = "/content/drive/MyDrive/Colab Notebooks/Аспирантура/conllu/handlabeled/GramEval2020-test-fiction.conllu"

with open(conllu_file, "r", encoding="utf-8") as f:
    conllu_data = f.read()


In [12]:
conllu_text = parse_conllu(conllu_data)
# print(conllu_text)

for token in conllu_text[0]:
  print(token)


{'form': 'С', 'lemma': 'с', 'upos': 'ADP', 'UD_lemma': 'с_PR', 'UD_pos': 'PR', 'feats': None, 'head': 3, 'deprel': 'case', 'sem': []}
{'form': 'некоторых', 'lemma': 'некоторый', 'upos': 'DET', 'UD_lemma': 'некоторый_APRO', 'UD_pos': 'APRO', 'feats': {'Case': 'Gen', 'Number': 'Plur', 'PronType': 'Ind'}, 'head': 3, 'deprel': 'det', 'sem': []}
{'form': 'пор', 'lemma': 'пора', 'upos': 'NOUN', 'UD_lemma': 'пора_S', 'UD_pos': 'S', 'feats': {'Animacy': 'Inan', 'Case': 'Gen', 'Gender': 'Fem', 'Number': 'Plur'}, 'head': 5, 'deprel': 'obl', 'sem': ['abstr', 'time']}
{'form': 'я', 'lemma': 'я', 'upos': 'PRON', 'UD_lemma': 'я_SPRO', 'UD_pos': 'SPRO', 'feats': {'Case': 'Nom', 'Number': 'Sing', 'Person': '1', 'PronType': 'Prs'}, 'head': 5, 'deprel': 'nsubj', 'sem': ['concr', 'hum']}
{'form': 'полюбил', 'lemma': 'полюбить', 'upos': 'VERB', 'UD_lemma': 'полюбить_V', 'UD_pos': 'V', 'feats': {'Aspect': 'Perf', 'Gender': 'Masc', 'Mood': 'Ind', 'Number': 'Sing', 'Tense': 'Past', 'Transit': 'Tran', 'VerbFo

## 2.3 Считаем вектор набора слов (в формате conllu): берем среднее векторов всех слов в предложении

In [13]:
import numpy as np

# на вход
# [{'form': 'С', 'lemma': 'с', 'upos': 'ADP', 'UD_lemma': 'с_PR', 'UD_pos': 'PR', 'feats': None, 'head': 3, 'deprel': 'case', 'sem': []}, {'form': 'некоторых', 'lemma': 'некоторый', 'upos': 'DET', 'UD_lemma': 'некоторый_APRO', 'UD_pos': 'APRO', 'feats': {'Case': 'Gen', 'Number': 'Plur', 'PronType': 'Ind'}, 'head': 3, 'deprel': 'det', 'sem': []}, {'form': 'пор', 'lemma': 'пора', 'upos': 'NOUN', 'UD_lemma': 'пора_S', 'UD_pos': 'S', 'feats': {'Animacy': 'Inan', 'Case': 'Gen', 'Gender': 'Fem', 'Number': 'Plur'}, 'head': 5, 'deprel': 'obl', 'sem': ['abstr', 'time']}, {'form': 'я', 'lemma': 'я', 'upos': 'PRON', 'UD_lemma': 'я_SPRO', 'UD_pos': 'SPRO', 'feats': {'Case': 'Nom', 'Number': 'Sing', 'Person': '1', 'PronType': 'Prs'}, 'head': 5, 'deprel': 'nsubj', 'sem': ['concr', 'hum']}, {'form': 'полюбил', 'lemma': 'полюбить', 'upos': 'VERB', 'UD_lemma': 'полюбить_V', 'UD_pos': 'V', 'feats': {'Aspect': 'Perf', 'Gender': 'Masc', 'Mood': 'Ind', 'Number': 'Sing', 'Tense': 'Past', 'Transit': 'Tran', 'VerbForm': 'Fin', 'Voice': 'Act'}, 'head': 0, 'deprel': 'root', 'sem': []}, {'form': 'это', 'lemma': 'этот', 'upos': 'DET', 'UD_lemma': 'этот_APRO', 'UD_pos': 'APRO', 'feats': {'Case': 'Acc', 'Gender': 'Neut', 'Number': 'Sing', 'PronType': 'Dem'}, 'head': 8, 'deprel': 'det', 'sem': []}, {'form': 'небольшое', 'lemma': 'небольшой', 'upos': 'ADJ', 'UD_lemma': 'небольшой_A', 'UD_pos': 'A', 'feats': {'Case': 'Acc', 'Degree': 'Pos', 'Gender': 'Neut', 'Number': 'Sing'}, 'head': 8, 'deprel': 'amod', 'sem': ['min', 'physq', 'size']}, {'form': 'кафе', 'lemma': 'кафе', 'upos': 'NOUN', 'UD_lemma': 'кафе_S', 'UD_pos': 'S', 'feats': {'Animacy': 'Inan', 'Case': 'Acc', 'Gender': 'Neut', 'InflClass': 'Ind', 'Number': 'Sing'}, 'head': 5, 'deprel': 'obj', 'sem': ['concr', 'org']}, {'form': '.', 'lemma': '.', 'upos': 'PUNCT', 'UD_lemma': '.', 'UD_pos': None, 'feats': None, 'head': 5, 'deprel': 'punct', 'sem': []}]
def conllu_text_to_vector(parsed_text, model):
    vectors = []

    for token in parsed_text:
        # Пропускаем пунктуацию и символы
        if token["upos"] in ["SYM", "PUNCT"]:
            continue

        # Проверяем наличие токена в Word2Vec
        if token["UD_lemma"] in model.wv.key_to_index:
            vectors.append(
                model.wv[token["UD_lemma"]]
            )

    # Если ни одного слова нет в модели
    if not vectors:
        return None

    return np.mean(vectors, axis=0)

    """
     Возвращает:
        np.ndarray размерности model.vector_size
        или None, если ни одного токена нет в словаре модели.
    """


In [14]:
print(conllu_text[0])


[{'form': 'С', 'lemma': 'с', 'upos': 'ADP', 'UD_lemma': 'с_PR', 'UD_pos': 'PR', 'feats': None, 'head': 3, 'deprel': 'case', 'sem': []}, {'form': 'некоторых', 'lemma': 'некоторый', 'upos': 'DET', 'UD_lemma': 'некоторый_APRO', 'UD_pos': 'APRO', 'feats': {'Case': 'Gen', 'Number': 'Plur', 'PronType': 'Ind'}, 'head': 3, 'deprel': 'det', 'sem': []}, {'form': 'пор', 'lemma': 'пора', 'upos': 'NOUN', 'UD_lemma': 'пора_S', 'UD_pos': 'S', 'feats': {'Animacy': 'Inan', 'Case': 'Gen', 'Gender': 'Fem', 'Number': 'Plur'}, 'head': 5, 'deprel': 'obl', 'sem': ['abstr', 'time']}, {'form': 'я', 'lemma': 'я', 'upos': 'PRON', 'UD_lemma': 'я_SPRO', 'UD_pos': 'SPRO', 'feats': {'Case': 'Nom', 'Number': 'Sing', 'Person': '1', 'PronType': 'Prs'}, 'head': 5, 'deprel': 'nsubj', 'sem': ['concr', 'hum']}, {'form': 'полюбил', 'lemma': 'полюбить', 'upos': 'VERB', 'UD_lemma': 'полюбить_V', 'UD_pos': 'V', 'feats': {'Aspect': 'Perf', 'Gender': 'Masc', 'Mood': 'Ind', 'Number': 'Sing', 'Tense': 'Past', 'Transit': 'Tran', 'V

In [15]:
print(conllu_text_to_vector(conllu_text[0], model))


NameError: name 'model' is not defined

## 2.3 (Дополнительно) Собираем все употребления тега

In [ ]:
def extract_examples(conllu_text, target_tags=None):
    sentences = parse(conllu_text)
    results = []

    if target_tags:
        target_tags = [t.strip().lower() for t in target_tags]

    for sent in sentences:
        sentence_text = sent.metadata.get("text", "")

        for token in sent:
            sem_tags = extract_sem_tags(token.get("misc"))
            sem_tags = [t.lower() for t in sem_tags]

            if not sem_tags:
                continue

            if target_tags:
                if not any(tag in sem_tags for tag in target_tags):
                    continue

            results.append({
                "form": token["form"],
                "lemma": token["lemma"],
                "upos": token["upos"],
                "sem": sem_tags,
                "sentence": sentence_text,
                "token_id": token["id"]
            })

    return results


In [ ]:
target_tags = ["tool"]

searching_for_tags = extract_examples(conllu_data, target_tags)
for found_word_with_tag in searching_for_tags:
  print(found_word_with_tag)


{'form': 'автомат', 'lemma': 'автомат', 'upos': 'NOUN', 'sem': ['concr', 'device', 'tool'], 'sentence': 'Может быть, потому, что музыкальный автомат грохотал здесь не так оглушительно, как везде.', 'token_id': 8}
{'form': 'столик', 'lemma': 'столик', 'upos': 'NOUN', 'sem': ['concr', 'furn', 'tool'], 'sentence': 'Я приходил сюда каждый вечер и садился за угловой столик, откуда, слава богу, не виден был экран телевизора.', 'token_id': 10}
{'form': 'экран', 'lemma': 'экран', 'upos': 'NOUN', 'sem': ['concr', 'device', 'part', 'tool'], 'sentence': 'Я приходил сюда каждый вечер и садился за угловой столик, откуда, слава богу, не виден был экран телевизора.', 'token_id': 20}
{'form': 'телевизора', 'lemma': 'телевизор', 'upos': 'NOUN', 'sem': ['concr', 'device', 'tool'], 'sentence': 'Я приходил сюда каждый вечер и садился за угловой столик, откуда, слава богу, не виден был экран телевизора.', 'token_id': 21}
{'form': 'дверь', 'lemma': 'дверь', 'upos': 'NOUN', 'sem': ['concr', 'constr', 'part',

Попробуем сначала с контекстом во все предложение

# Cross-Encoder

(context, gloss)

↓

BERT

↓

binary relevance score

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL = "DeepPavlov/rubert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you